In [ ]:
import torch
#修正原文中返回标量的reward,这里应该返回和x形状相同的梯度
def collision_guidance_fn(x, t, cond, inputs, *args, **kwargs) -> torch.Tensor:
    # ... 前面的碰撞检测计算保持不变 ...
    
    # 计算碰撞损失
    distances = batch_signed_distance_rect(ego_bbox, neighbor_bbox)
    clip_distances = torch.maximum(1 - distances / CLIP_DISTANCE, torch.tensor(0.0, device=distances.device))
    
    reward = - (torch.sum(clip_distances[clip_distances > 1]) / (torch.sum((clip_distances[clip_distances > 1].detach() > 0).float()) + 1e-5) +
                torch.sum(clip_distances[clip_distances <= 1]) / (torch.sum((clip_distances[clip_distances <= 1].detach() > 0).float()) + 1e-5)).exp()
    
    # 关键修改：返回完整梯度，不进行切片
    grad_x = torch.autograd.grad(reward.sum(), x, retain_graph=True, allow_unused=True)[0]
    
    # 可选：对梯度进行后处理
    if grad_x is not None:
        # 梯度裁剪，避免数值不稳定
        grad_x = torch.clamp(grad_x, -1.0, 1.0)
        
        # 只在有效时间应用引导
        mask_diffusion_time = (t < 0.1) & (t > 0.005)
        grad_x = torch.where(mask_diffusion_time.view(B, 1, 1, 1), grad_x, torch.zeros_like(grad_x))
        
        return grad_x  # [B, P, T, 4]
    else:
        return torch.zeros_like(x)